In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os


class SimpleAttention(nn.Module):
    """
    A simple, ONNX-traceable multi-head attention module.
    """
    def __init__(self, hidden_dim, n_head):
        super().__init__()
        assert hidden_dim % n_head == 0, "hidden_dim must be divisible by n_head"
        
        self.n_head = n_head
        self.head_dim = hidden_dim // n_head
        self.scale = self.head_dim ** -0.5
        
        # Use a single linear layer for combined Q, K, V projections for efficiency
        self.in_proj = nn.Linear(hidden_dim, hidden_dim * 3, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, query, key, value):
        # Input shapes: (seq_len, batch_size, hidden_dim)
        seq_len, batch_size, _ = query.shape

        # 1. Linearly project and split Q, K, V
        # self.in_proj(query) -> (seq_len, batch_size, hidden_dim * 3)
        # .chunk(3, dim=-1) splits it into 3 tensors of shape (seq_len, batch_size, hidden_dim)
        q, k, v = self.in_proj(query).chunk(3, dim=-1)

        # 2. Reshape and permute for multi-head processing
        # (seq_len, batch, hidden_dim) -> (seq_len, batch, n_head, head_dim)
        q = q.view(seq_len, batch_size, self.n_head, self.head_dim)
        k = k.view(seq_len, batch_size, self.n_head, self.head_dim)
        v = v.view(seq_len, batch_size, self.n_head, self.head_dim)
        
        # (seq_len, batch, n_head, head_dim) -> (batch, n_head, seq_len, head_dim)
        # This groups batches and heads together for matrix multiplication
        q = q.permute(1, 2, 0, 3)
        k = k.permute(1, 2, 0, 3)
        v = v.permute(1, 2, 0, 3)

        # 3. Scaled dot-product attention
        # (batch, n_head, seq_len, head_dim) @ (batch, n_head, head_dim, seq_len)
        # gives scores of shape (batch, n_head, seq_len, seq_len)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn_scores, dim=-1)
        
        # (batch, n_head, seq_len, seq_len) @ (batch, n_head, seq_len, head_dim)
        # gives output of shape (batch, n_head, seq_len, head_dim)
        attn_output = torch.matmul(attn_weights, v)

        # 4. Concatenate heads and apply final linear projection
        # (batch, n_head, seq_len, head_dim) -> (seq_len, batch, n_head, head_dim)
        attn_output = attn_output.permute(2, 0, 1, 3)
        # .contiguous() is important before .view() after a permute
        # (seq_len, batch, n_head, head_dim) -> (seq_len, batch, hidden_dim)
        attn_output = attn_output.contiguous().view(seq_len, batch_size, -1)
        output = self.out_proj(attn_output)
        return output


class ToyLocalTransformer(nn.Module):
    def __init__(self, in_dim=16192, hidden_dim=768, vocab_size=2024, n_head=1):
        super().__init__()
        self.cfg_scale = 2.5
        self.embedding = nn.Embedding(in_dim, hidden_dim)
        self.in_proj = nn.Linear(in_dim, hidden_dim, bias=False)
        self.attn = SimpleAttention(hidden_dim, n_head)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, hidden_states, tokens):
        # hidden states have shape (batch_size, hidden_dim)
        # tokens have shape (N, batch_size'), where N are tokens from previous local transformer steps
        # keep in mind that N is 0 for the first iteration
        token_embs = self.embedding(tokens.transpose(1, 0))  # b' x N x dim
        # Repeat each batch element along the batch dimension so that 0 and 1 are the same, 2 and 3, etc.
        token_embs = token_embs.repeat_interleave(2, dim=0)  # 2b x N x dim
        hidden_states_proj = self.in_proj(hidden_states)  # 2 x dim
        token_embs[:, 0, :] += hidden_states_proj  # b x N x dim
        
        # simplified transformer block
        # MultiheadAttention expects shape (seq_len, batch, dim)
        x_attn = token_embs.transpose(0, 1)
        x_attn = self.attn(x_attn, x_attn, x_attn)
        # revert back to (batch, seq_len, dim)
        x_attn = x_attn.transpose(0, 1)
        x = token_embs + x_attn

        logits = self.lm_head(x)  # b x (N + 1) x vocab_size
        # select only last logits
        last_logits = logits[:, -1, :]  # b x vocab_size

        # Alternative approach: use slicing with explicit stride
        cond_logits = last_logits[::2, :]  # select even indices (0,2,4,...)
        uncond_logits = last_logits[1::2, :]  # select odd indices (1,3,5,...)
        final_logits = cond_logits * self.cfg_scale + uncond_logits * (1 - self.cfg_scale)
        return final_logits  # b' x vocab_size

In [ ]:
    # --- Configuration ---
onnx_file_path = "local_transformer.onnx"

# --- Model Initialization ---
print("Initializing models...")
lt = ToyLocalTransformer()
lt.eval().cuda().half()

DUMMY_HALF_BATCH_SIZE = 2
DUMMY_TOKENS_NUM = 3
dummy_hidden_state = torch.randn((DUMMY_HALF_BATCH_SIZE * 2, 16192)).cuda().half()
dummy_tokens = torch.zeros((DUMMY_TOKENS_NUM, DUMMY_HALF_BATCH_SIZE), dtype=torch.int).cuda()

# --- ONNX Export ---
print(f"\nAttempting to export the model to '{onnx_file_path}'...")

torch.onnx.export(
    lt, 
    (dummy_hidden_state, dummy_tokens), 
    onnx_file_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['hidden_states', 'tokens'],
    output_names=['logits'],
    dynamic_axes={
        'hidden_states': {0: 'batch_size'},
        'tokens': {0: 'tokens_num', 1: 'half_batch_size'},
        'logits': {0: 'half_batch_size'}
    }
)
print(f"File saved at: {os.path.abspath(onnx_file_path)}")

In [ ]:
import onnxruntime as ort
import numpy as np

ort_session = ort.InferenceSession(onnx_file_path, providers=['CPUExecutionProvider'])

bs = 2
num_tok = 8
hs = np.random.randn(bs * 2, 16192).astype(np.float16)
toks = np.zeros((num_tok, bs), dtype=np.int32)
outputs = ort_session.run(None, {"hidden_states": hs, "tokens": toks})
print(outputs[0].shape)

In [ ]:
# --fp16
CUDA_VISIBLE_DEVICES=0 /usr/local/tensorrt/targets/x86_64-linux-gnu/bin/trtexec --fp16 \
    --onnx=/code/tensorrt_llm/local_transformer.onnx \
    --saveEngine=/code/tensorrt_llm/local_transformer.trt \
    --minShapes=hidden_states:2x16192,tokens:1x1  \
    --optShapes=hidden_states:8x16192,tokens:4x4 \
    --maxShapes=hidden_states:32x16192,tokens:8x16